In [13]:
from Bio import SeqIO
import glob, os
import numpy as np
import pandas as pd
import torch

In [14]:
# Fonction load_labels + load_pt
PT_DIR   = "embeddings_out"
GBK_DIR  = "genomes_gbff"
RGI_DIR  = "rgi_results"
LABELS_PATH = "matrice_listantibio_final.cleaned.list"
 
def load_labels(path):
    df = pd.read_csv(path, sep=None, engine="python")
    df["#Genome_ID_Genome_name"] = df["#Genome_ID_Genome_name"].astype(str)
    df["Resistant"]   = df["Resistant"].astype(int)
    df["Susceptible"] = df["Susceptible"].astype(int)
    return dict(zip(df["#Genome_ID_Genome_name"], df["Resistant"]))
 
def load_pt(fp):
    obj = torch.load(fp)
    gid = str(obj.get("genome_id", os.path.splitext(os.path.basename(fp))[0]))
    X   = obj["last_hidden_state"]
    if isinstance(X, np.ndarray):
        X = torch.from_numpy(X)
    if X.ndim == 3:
        X = X.squeeze(0)
    if X.ndim != 2:
        raise ValueError(f"{fp}: shape inattendue {tuple(X.shape)}")
    return gid, X.float()
 
# Charge labels + fichiers
id_to_y   = load_labels(LABELS_PATH)
all_files = sorted(glob.glob(os.path.join(PT_DIR, "*.pt")))
matched   = [fp for fp in all_files if load_pt(fp)[0] in id_to_y]
resistant_files  = [fp for fp in matched if id_to_y[load_pt(fp)[0]] == 1]
susceptible_files= [fp for fp in matched if id_to_y[load_pt(fp)[0]] == 0]
 
print(f"Labels chargés")
print(f"Total .pt matchés  : {len(matched)}")
print(f"Résistants         : {len(resistant_files)}")
print(f"Sensibles          : {len(susceptible_files)}")

Labels chargés
Total .pt matchés  : 907
Résistants         : 53
Sensibles          : 854


In [17]:
# IDs dans tes .pt
pt_ids  = set(os.path.splitext(os.path.basename(f))[0]
              for f in glob.glob("embeddings_out/*.pt"))

# IDs dans tes résultats RGI
rgi_ids = set(os.path.splitext(os.path.basename(f))[0]
              for f in glob.glob("rgi_results/*.txt"))

# IDs dans tes .gbff
gbff_ids = set(os.path.splitext(os.path.basename(f))[0]
               for f in glob.glob("genomes_gbff/*.gbff"))

print(f"IDs dans .pt   : {len(pt_ids)}")
print(f"IDs dans RGI   : {len(rgi_ids)}")
print(f"IDs dans .gbff : {len(gbff_ids)}")

IDs dans .pt   : 1066
IDs dans RGI   : 1066
IDs dans .gbff : 1066


In [19]:
# Parser le .gbff pour extraire les CDS dans l'ordre
 
def load_gbff_annotation(genome_id):
    gbff_file = os.path.join(GBK_DIR, f"{genome_id}.gbff")
    if not os.path.exists(gbff_file):
        print(f"Fichier non trouvé : {gbff_file}")
        return None
    rows = []
    for record in SeqIO.parse(gbff_file, "genbank"):
        for feature in record.features:
            if feature.type != "CDS":
                continue
            rows.append({
                "contig":    record.id,
                "start":     int(feature.location.start),
                "end":       int(feature.location.end),
                "strand":    int(feature.location.strand),
                "locus_tag": feature.qualifiers.get("locus_tag", [""])[0],
                "gene":      feature.qualifiers.get("gene",      [""])[0],
                "product":   feature.qualifiers.get("product",   ["unknown"])[0],
            })
    if not rows:
        print(f"Aucun CDS trouvé")
        return None
    df = pd.DataFrame(rows).reset_index()
    return df.rename(columns={"index": "protein_index"})
 
# Test sur le premier génome disponible
sample_gid = load_pt(all_files[0])[0]
ann_test   = load_gbff_annotation(sample_gid)
 
if ann_test is not None:
    print(f".gbff chargé pour : {sample_gid}")
    print(f"   Nb CDS : {len(ann_test)}")
    print(ann_test[["protein_index","gene","product","start","end"]].head(10))
else:
    print("Problème — vérifie GBK_DIR et le nom des fichiers")

.gbff chargé pour : 1328432.3
   Nb CDS : 4665
   protein_index    gene                                            product  \
0              0    thrA  Bifunctional aspartokinase/homoserine dehydrog...   
1              1    thrB                                  Homoserine kinase   
2              2    thrC                                 Threonine synthase   
3              3                                       hypothetical protein   
4              4    yaaA  DNA-binding and peroxide stress resistance pro...   
5              5    alsT                    Amino-acid carrier protein AlsT   
6              6  talB_1                                    Transaldolase B   
7              7     mog                  Molybdopterin adenylyltransferase   
8              8    satP            Succinate-acetate/proton symporter SatP   
9              9                                       hypothetical protein   

   start    end  
0    335   2798  
1   2799   3732  
2   3732   5019  
3   5232   

In [21]:
# Cellule diagnostic — affiche les colonnes exactes de ton fichier RGI
import pandas as pd, os

test_gid = load_pt(resistant_files[0])[0]
rgi_file = os.path.join(RGI_DIR, f"{test_gid}.txt")

df = pd.read_csv(rgi_file, sep="\t")
print(f"Colonnes disponibles :")
for col in df.columns:
    print(f"  → '{col}'")

print(f"\nPremières lignes :")
print(df.head(3).to_string())

Colonnes disponibles :
  → 'ORF_ID'
  → 'Contig'
  → 'Start'
  → 'Stop'
  → 'Orientation'
  → 'Cut_Off'
  → 'Pass_Bitscore'
  → 'Best_Hit_Bitscore'
  → 'Best_Hit_ARO'
  → 'Best_Identities'
  → 'ARO'
  → 'Model_type'
  → 'SNPs_in_Best_Hit_ARO'
  → 'Other_SNPs'
  → 'Drug Class'
  → 'Resistance Mechanism'
  → 'AMR Gene Family'
  → 'Predicted_DNA'
  → 'Predicted_Protein'
  → 'CARD_Protein_Sequence'
  → 'Percentage Length of Reference Sequence'
  → 'ID'
  → 'Model_ID'
  → 'Nudged'
  → 'Note'
  → 'Hit_Start'
  → 'Hit_End'
  → 'Antibiotic'
  → 'AST_Source'

Premières lignes :
                                                                                                                               ORF_ID        Contig   Start    Stop Orientation  Cut_Off  Pass_Bitscore  Best_Hit_Bitscore           Best_Hit_ARO  Best_Identities      ARO             Model_type SNPs_in_Best_Hit_ARO Other_SNPs                                                                                                      

In [25]:
# Parser les résultats RGI
 
def load_rgi_meropenem(genome_id):
    rgi_file = os.path.join(RGI_DIR, f"{genome_id}.txt")
    if not os.path.exists(rgi_file):
        return None

    df = pd.read_csv(rgi_file, sep="\t")

    # Garde Perfect + Strict uniquement
    df = df[df["Cut_Off"].isin(["Perfect", "Strict"])].copy()
    if len(df) == 0:
        return None

    # ── Filtre méropénème sur "Drug Class" ───
    mask = df["Drug Class"].str.lower().str.contains(
        "carbapenem|penem", na=False
    )
    df = df[mask].copy()
    if len(df) == 0:
        return None

    # Parse start/stop depuis Start/Stop
    df["rgi_start"] = df["Start"].astype(int)
    df["rgi_stop"]  = df["Stop"].astype(int)

    return df.reset_index(drop=True)

# Test sur le premier génome résistant
test_gid = load_pt(resistant_files[0])[0]
rgi_test = load_rgi_meropenem(test_gid)

print(f"Test RGI méropénème sur : {test_gid}")
if rgi_test is not None:
    print(f"   Nb gènes carbapénème détectés : {len(rgi_test)}")
    print(rgi_test[["Best_Hit_ARO", "Drug Class", "Cut_Off", "Best_Identities"]].to_string(index=False))
else:
    print("Aucun gène carbapénème trouvé")
    # Diagnostic — affiche tous les Drug Class trouvés
    df_all = pd.read_csv(os.path.join(RGI_DIR, f"{test_gid}.txt"), sep="\t")
    df_all = df_all[df_all["Cut_Off"].isin(["Perfect","Strict"])]
    print(f"\n   Tous les Drug Class trouvés :")
    for dc in df_all["Drug Class"].unique():
        print(f"     → {dc}")

Test RGI méropénème sur : 1328437.3
   Nb gènes carbapénème détectés : 3
                                                        Best_Hit_ARO                                                                                                                                                                                                        Drug Class Cut_Off  Best_Identities
                                                                marA fluoroquinolone antibiotic; monobactam; carbapenem; cephalosporin; glycylcycline; penicillin beta-lactam; tetracycline antibiotic; rifamycin antibiotic; phenicol antibiotic; disinfecting agents and antiseptics Perfect            100.0
                                                               KPC-2                                                                                                                                                     monobactam; carbapenem; cephalosporin; penicillin beta-lactam Perfect            100.0
Escherichia col

In [28]:
# Croiser RGI + .gbff → trouver l'indice protéine
 
def find_resistance_indices(genome_id):
    ann    = load_gbff_annotation(genome_id)
    rgi_df = load_rgi_meropenem(genome_id)
    if ann is None or rgi_df is None:
        return []
    results = []
    for _, rgi_gene in rgi_df.iterrows():
        rgi_start = int(rgi_gene["rgi_start"])
        rgi_stop  = int(rgi_gene["rgi_stop"])
        best_idx, best_overlap, best_row = None, 0, None
        for _, cds in ann.iterrows():
            overlap = max(0,
                min(rgi_stop, cds["end"]) - max(rgi_start, cds["start"]))
            if overlap > best_overlap:
                best_overlap = overlap
                best_idx     = int(cds["protein_index"])
                best_row     = cds
        if best_idx is not None and best_row is not None:
            results.append({
                "protein_index":        best_idx,
                "aro_name":             rgi_gene.get("Best_Hit_ARO",         ""),
                "drug_class":           rgi_gene.get("Drug Class",           ""),  
                "resistance_mechanism": rgi_gene.get("Resistance Mechanism", ""),  
                "cut_off":              rgi_gene.get("Cut_Off",              ""),
                "identity":             rgi_gene.get("Best_Identities",      0),
                "gene":                 best_row["gene"],
                "product":              best_row["product"],
                "locus_tag":            best_row["locus_tag"],
                "overlap_bp":           best_overlap,
            })
    return results

# Test sur le premier génome résistant
res_genes = find_resistance_indices(test_gid)

print(f"Croisement RGI + .gbff pour : {test_gid}\n")
if res_genes:
    for g in res_genes:
        print(f"   → {g['aro_name']:40s} | indice : {g['protein_index']:5d} | "
              f"id : {g['identity']:.1f}% | {g['drug_class'][:40]}")
else:
    print("Aucun gène méropénème trouvé")

Croisement RGI + .gbff pour : 1328437.3

   → marA                                     | indice :   608 | id : 100.0% | fluoroquinolone antibiotic; monobactam; 
   → KPC-2                                    | indice :  2531 | id : 100.0% | monobactam; carbapenem; cephalosporin; p
   → Escherichia coli soxS with mutation conferring antibiotic resistance | indice :  1908 | id : 100.0% | fluoroquinolone antibiotic; monobactam; 


In [29]:
# RGI méropénème sur TOUS les génomes + stockage

import json
from tqdm import tqdm

all_res_genes = {}   
stats = {"avec_genes": 0, "sans_genes": 0, "sans_rgi": 0, "sans_gbff": 0}

print(f"Analyse de {len(matched)} génomes...\n")

for fp in tqdm(matched):
    gid = load_pt(fp)[0]

    # Vérifie que les fichiers existent
    rgi_file  = os.path.join(RGI_DIR,  f"{gid}.txt")
    gbff_file = os.path.join(GBK_DIR,  f"{gid}.gbff")

    if not os.path.exists(rgi_file):
        stats["sans_rgi"] += 1
        continue
    if not os.path.exists(gbff_file):
        stats["sans_gbff"] += 1
        continue

    # Trouve les gènes de résistance méropénème
    res_genes = find_resistance_indices(gid)

    if res_genes:
        all_res_genes[gid] = res_genes
        stats["avec_genes"] += 1
    else:
        all_res_genes[gid] = []
        stats["sans_genes"] += 1

# Résumé 
print(f"\n Résumé :")
print(f"   Génomes analysés              : {len(matched)}")
print(f"   Avec gènes carbapénème        : {stats['avec_genes']}")
print(f"   Sans gènes carbapénème        : {stats['sans_genes']}")
print(f"   Sans fichier RGI              : {stats['sans_rgi']}")
print(f"   Sans fichier .gbff            : {stats['sans_gbff']}")

# Gènes les plus fréquents
from collections import Counter
all_gene_names = [g["aro_name"] for genes in all_res_genes.values() for g in genes]
gene_counts    = Counter(all_gene_names).most_common(15)

print(f"\n Top 15 gènes carbapénème les plus fréquents :")
for gene, count in gene_counts:
    bar = "█" * int(count / max(gene_counts[0][1], 1) * 20)
    print(f"   {gene:45s} | {count:4d} génomes  {bar}")

# Résistants vs sensibles
print(f"\n Gènes trouvés selon le phénotype :")
for label, label_name in [(1, "Résistants"), (0, "Sensibles")]:
    gids_label = [load_pt(fp)[0] for fp in matched if id_to_y[load_pt(fp)[0]] == label]
    genes_label = [g["aro_name"] for gid in gids_label
                   if gid in all_res_genes
                   for g in all_res_genes[gid]]
    counts = Counter(genes_label).most_common(5)
    print(f"\n   {label_name} — top 5 gènes :")
    for gene, count in counts:
        print(f"     → {gene:40s} : {count} génomes")

# Sauvegarde JSON
with open("resistance_genes_all.json", "w") as f:
    json.dump(all_res_genes, f, indent=2)
print(f"\n Sauvegardé → resistance_genes_all.json")

# Sauvegarde CSV
rows = []
for gid, genes in all_res_genes.items():
    for g in genes:
        rows.append({
            "genome_id":  gid,
            "true_label": id_to_y.get(gid, -1),
            **g
        })

df_all_genes = pd.DataFrame(rows)
df_all_genes.to_csv("resistance_genes_all.csv", index=False)
print(f" Sauvegardé → resistance_genes_all.csv ({len(df_all_genes)} lignes)")
print(df_all_genes[["genome_id","true_label","aro_name",
                     "protein_index","identity","cut_off"]].head(20).to_string(index=False))

Analyse de 907 génomes...



100%|██████████| 907/907 [18:02<00:00,  1.19s/it]



 Résumé :
   Génomes analysés              : 907
   Avec gènes carbapénème        : 899
   Sans gènes carbapénème        : 8
   Sans fichier RGI              : 0
   Sans fichier .gbff            : 0

 Top 15 gènes carbapénème les plus fréquents :
   Escherichia coli soxS with mutation conferring antibiotic resistance |  892 génomes  ████████████████████
   marA                                          |  884 génomes  ███████████████████
   TolC                                          |  548 génomes  ████████████
   CMY-2                                         |  172 génomes  ███
   CTX-M-27                                      |   27 génomes  
   KPC-2                                         |   16 génomes  
   KPC-3                                         |    9 génomes  
   NDM-1                                         |    8 génomes  
   NDM-5                                         |    8 génomes  
   Klebsiella pneumoniae KpnG                    |    4 génomes  
   Klebsiella p